# 矩阵分形布局

上一节的 `load → move → matmul → store` 中，逻辑矩阵形状始终可见，物理元素顺序却随 Buffer 改变。本节先用一个能逐个数出元素的小矩阵解释 NZ/ZN，再回到真实 FP16 分形，最后完成一个接收转置 B 输入的矩阵乘。


## 环境准备

运行代码前，请完成 [01.06 环境配置](../01_pypto_pro_introduction/01.06_env_and_first_kernel.ipynb)。

在本节所在目录启动 Notebook，按顺序执行单元即可；不需要先运行其他 Notebook。设备号默认取 `TILE_FWK_DEVICE_ID`，未设置时使用 0。


In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

device = f"npu:{int(os.environ['TILE_FWK_DEVICE_ID'])}"
torch.npu.set_device(device)


## 1. 同一矩阵，为什么需要不同的排列

GM 中常见的二维 ND 输入按行连续存放。Cube 消费的是二维分形块，搬运指令需要把跨行的数据整理成硬件所需的排列。分形描述的是“元素放在哪里”，矩阵的数学含义仍由逻辑坐标决定。

例如逻辑 `B[K,N]` 从 L1 移到 L0B 时，默认由 NZ 变成 ZN，但仍然是同一个 `B[K,N]`，没有自动变成数学上的 `B.T`。如果把布局变化误解为转置，就会把 K、N 轴也写反。
<img style="display:block; margin-left:0; margin-right:auto;" src="./images/cube_matrix_nd_to_nz.svg" alt="40×56 的有效矩阵与 48×64 分形边界" width="700px">


图中的白色区域对应有效输入，灰色区域是对齐形成的无效区；它们的值不能凭颜色推定为零。我们先用一个更小的例子弄清元素顺序，再计算图中的存储占用。


## 2. NZ / ZN：先看块之间，再看块内部

“大 Y 小 x”的记法把排列分成两层：

- 大写位置描述**分形之间**的排列：N 为列优先，Z 为行优先。
- 小写位置描述**分形内部**的排列：z 为行优先，n 为列优先。API 的枚举名统一写成 `NZ`、`ZN`。

为了能手算，暂时把分形缩成 `2×2`。**这只是排列示意，不能把 2×2 当成 950 的硬件分形规格。** 给一个 `4×4` 矩阵按行编号：

```text
          列 0  1     列 2  3
行 0       0  1   |    2  3
行 1       4  5   |    6  7
          -------+--------
行 2       8  9   |   10 11
行 3      12 13   |   14 15
```

- ND：直接逐行读取，得到 `0,1,2,3,4,5,...,15`。
- NZ：块按“左上 → 左下 → 右上 → 右下”读取，每块内部逐行读取。
- ZN：块按“左上 → 右上 → 左下 → 右下”读取，每块内部逐列读取。

因此三种存储顺序为：

```text
ND:  0  1  2  3 |  4  5  6  7 |  8  9 10 11 | 12 13 14 15
NZ:  0  1  4  5 |  8  9 12 13 |  2  3  6  7 | 10 11 14 15
ZN:  0  4  1  5 |  2  6  3  7 |  8 12  9 13 | 10 14 11 15
```

例如值 9 的逻辑坐标始终是 `[2,1]`，但从零开始计数时，它在 ND/NZ/ZN 中的元素偏移分别是 9、5、10。**逻辑坐标不变，物理地址改变**，这就是格式转换的含义。


### 把排列写成索引公式

对已补齐的 `[M,N]`，设分形大小为 `[M0,N0]`，块数为 `M1=M/M0`、`N1=N/N0`。逻辑 `[m,n]` 拆为 `(m1,m0,n1,n0)`：

- NZ 的物理轴序为 `[N1,M1,M0,N0]`，元素偏移为 `((n1*M1+m1)*M0+m0)*N0+n0`。
- ZN 的物理轴序为 `[M1,N1,N0,M0]`，元素偏移为 `((m1*N1+n1)*N0+n0)*M0+m0`。

下面仅用 Python 列表核对上面的教学示意，不执行也不模拟 Device Kernel，可独立运行此单元：


In [ ]:
x = [[4 * row + col for col in range(4)] for row in range(4)]
nz = [x[2 * br + r][2 * bc + c]
      for bc in range(2) for br in range(2)
      for r in range(2) for c in range(2)]
zn = [x[2 * br + r][2 * bc + c]
      for br in range(2) for bc in range(2)
      for c in range(2) for r in range(2)]
assert nz == [0, 1, 4, 5, 8, 9, 12, 13, 2, 3, 6, 7, 10, 11, 14, 15]
assert zn == [0, 4, 1, 5, 2, 6, 3, 7, 8, 12, 9, 13, 10, 14, 11, 15]
assert nz.index(9) == 5 and zn.index(9) == 10
print('NZ:', nz)
print('ZN:', zn)


## 3. 回到真实分形：规格与容量

对下表中的常规输入类型，分形的一边为 16，另一边为 `32B / sizeof(dtype)`。例如 FP16 每个元素 2 B，因而另一边是 16；FP32 每个元素 4 B，另一边是 8：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">数据类型</th><th style="text-align:left;">A 矩阵分形</th><th style="text-align:left;">B 矩阵分形</th></tr>
<tr><td>FP32</td><td>16×8</td><td>8×16</td></tr>
<tr><td>FP16 / BF16</td><td>16×16</td><td>16×16</td></tr>
<tr><td>FP8（E4M3FN/E5M2）/ HF8 / INT8</td><td>16×32</td><td>32×16（累加 FP32/INT32）</td></tr>
</table>

L0C 中的**结果分形固定 16×16**（FP32 累加占 16×16×4B = 1024B）。


表中的输入类型必须组成 `matmul` 支持的组合，并不是任意 A/B 类型都能混用。本章的运行示例始终采用 FP16×FP16→FP32。

### 40×56 的有效数据需要预留多少空间？

按 FP16 的 `16×16` 分形对齐：

- 行边界：`ceil(40/16)×16 = 48`；列边界：`ceil(56/16)×16 = 64`。
- 有效数据：`40×56×2 = 4480 B`。
- 按完整分形预留：`48×64×2 = 6144 B`，比有效数据多 `1664 B`。

因此规划这样的 Tile 时，应声明对齐的物理 `shape=[48,64]`，并用 `valid_shape=[40,56]` 表达有效范围。不要先给 `shape=[40,56]` 分配 4480 B，再假定框架会自动扩大这段地址；物理 shape 的对齐与地址预算都需要开发者负责。

有效范围外的内容是否填充，由相应的 padding 配置与操作决定。`valid_shape` 限制有效数据范围，不意味着无效区已清零；尤其沿 K 累加时，不能让未定义的填充值参与结果。04.04 的示例先固定 K=128，只处理 M/N 尾块。


## 4. 在数据路径上识别自动转换

下表对应 04.02 中没有转置的普通 ND 输入：
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">数据路径</th><th style="text-align:left;">物理布局变化</th><th style="text-align:left;">逻辑形状</th></tr>
<tr><td>A：GM → L1 → L0A</td><td>ND → NZ → NZ</td><td>始终是 [M,K]</td></tr>
<tr><td>B：GM → L1 → L0B</td><td>ND → NZ → ZN</td><td>始终是 [K,N]</td></tr>
<tr><td>C：L0C → GM</td><td>NZ → ND</td><td>始终是 [M,N]</td></tr>
</table>


<img style="display:block; margin-left:0; margin-right:auto;" src="./images/cube_matrix_fractal_formats_950.svg" alt="Cube 默认 NZ × ZN → NZ 分形路径" width="700px">


你无需在 Host 手动把输入改成 NZ 数组，也无需把 `TileType.shape` 改成四维。`shape` 保留逻辑轴，`layout` 描述片上排列；未显式指定布局时，框架根据 MemorySpace 推导默认值。


## 5. 转置输入：同时改变轴映射和 L1 布局

现在假设上游已经把右矩阵存成连续的 `B_t[N,K]`，我们仍要计算 `A[M,K] @ B[K,N]`。这里涉及真正的逻辑转置：L1 Tile 的第 0 轴对应 GM Tensor 的第 1 轴，第 1 轴对应 GM 的第 0 轴。

普通二维输入的搬入组合如下：
<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">操作数</th><th style="text-align:left;">GM Tensor shape</th><th style="text-align:left;">L1 Tile shape / layout</th><th style="text-align:left;">load 的 order</th></tr>
<tr><td>A，未转置</td><td>[M,K]</td><td>[M,K] / NZ</td><td>[0,1]（默认）</td></tr>
<tr><td>A，以转置形式存储</td><td>[K,M]</td><td>[M,K] / ZN</td><td>[1,0]</td></tr>
<tr><td>B，未转置</td><td>[K,N]</td><td>[K,N] / NZ</td><td>[0,1]（默认）</td></tr>
<tr><td>B，以转置形式存储</td><td>[N,K]</td><td>[K,N] / ZN</td><td>[1,0]</td></tr>
</table>


本例选 `M=64,K=128,N=32`，刻意使用非方阵，使 K/N 写反更容易被发现。对照 04.02，关键变化有三处：GM 的 B 参数变成 `[32,128]`；B 的 L1 Tile 仍为 `[128,32]`，但布局设为 `ZN`；搬入时加 `order=[1,0]`。L0B 仍然是 `[K,N] / ZN`。


In [ ]:
@pl.jit(auto_mutex=True)
def mm_transposed_b(
    a: pl.Tensor[[64, 128], pl.DT_FP16],
    b_t: pl.Tensor[[32, 128], pl.DT_FP16],
    out: pl.Tensor[[64, 32], pl.DT_FP32],
):
    a_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[64, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[64, 128]),
        addrs=0x00000, mutex_ids=[0])
    b_l1 = pl.make_tile_group(
        type=pl.TileType(shape=[128, 32], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Mat, valid_shape=[128, 32],
                         layout=pl.ZN),
        addrs=0x10000, mutex_ids=[1])
    a_left = pl.make_tile_group(
        type=pl.TileType(shape=[64, 128], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Left, valid_shape=[64, 128]),
        addrs=0x0000, mutex_ids=[2])
    b_right = pl.make_tile_group(
        type=pl.TileType(shape=[128, 32], dtype=pl.DT_FP16,
                         target_memory=pl.MemorySpace.Right, valid_shape=[128, 32]),
        addrs=0x0000, mutex_ids=[3])
    acc = pl.make_tile_group(
        type=pl.TileType(shape=[64, 32], dtype=pl.DT_FP32,
                         target_memory=pl.MemorySpace.Acc, valid_shape=[64, 32],
                         fractal=1024),
        addrs=0x0000, mutex_ids=[4])

    with pl.section_cube():
        cur_a = a_l1.current()
        cur_b = b_l1.current()
        al = a_left.current()
        br = b_right.current()
        ac = acc.current()
        pl.load(cur_a, a, [0, 0])
        pl.load(cur_b, b_t, [0, 0], order=[1, 0])
        pl.move(al, cur_a)
        pl.move(br, cur_b)
        pl.matmul(ac, al, br)
        pl.store(out, ac, [0, 0])


Host 端先生成数学上的 B，再用 `b.t().contiguous()` 构造实际传入的 B_t。`contiguous()` 使转置后的矩阵按其新轴序连续存储，避免把“非连续视图的 stride”与本节讨论的转置搬运混在一起。Golden 始终用原始 B。


In [ ]:
torch.manual_seed(42)
a = torch.randn(64, 128, device=device, dtype=torch.float16)
b = torch.randn(128, 32, device=device, dtype=torch.float16)
b_t = b.t().contiguous()
out = torch.full((64, 32), float('nan'), device=device, dtype=torch.float32)
assert b_t.shape == (32, 128) and b_t.is_contiguous()

mm_transposed_b[None, 1](a, b_t, out)
torch.npu.synchronize()
assert torch.isfinite(out).all().item()
torch.testing.assert_close(out, a.float() @ b.float(), rtol=1e-2, atol=1e-2)
print('转置 B 输入校验通过')


## 6. 课后练习

1. （填空）FP16 输入时，A/B 的分形大小分别为 ______、______，FP32 的 L0C 结果分形大小为 ______ 个元素、______ 字节。
2. （选择）NZ 中第一个字母 N 表示：A. 分形内列优先；B. 分形之间列优先；C. 矩阵已被转置。
3. （推导）用本节 2×2 示意分形，值 6 在 ND、NZ、ZN 中的元素偏移分别是多少？它的逻辑坐标是否改变？
4. （实践）有效 `[40,56]` FP16 输入应怎样规划物理 shape 与字节容量？若 GM 中的右矩阵存为 `[N,K]`，为什么只把 L1 的 layout 改成 ZN 仍不够？


In [ ]:
!cat ./answer/04.03_answer.txt


## 7. 本节小结

布局告诉我们元素怎样排列，逻辑 shape 告诉我们这些元素属于哪个矩阵轴。分形对齐决定地址预算，`valid_shape` 表达有效范围；转置输入还要用 `order` 显式建立轴映射。

下一节 [多核切分与动态 shape](./04.04_multicore_and_dynamic_shape.ipynb) 将把输出扩展到 500×300，推导每个核处理的区域以及最后一块的有效形状。
